# Case 2: Lending Club Classification Analysis

### Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np

## Prepare dataset

In [2]:
loans_df = pd.read_csv('https://raw.githubusercontent.com/amandeep0/IS451/main/data/loans.csv')
loans_df.head()

,CreditPolicy,Purpose,IntRate,Installment,LogAnnualInc,Dti,Fico,DaysWithCrLine,RevolBal,RevolUtil,InqLast6mths,Delinq2yrs,PubRec,NotFullyPaid
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,0
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,0
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,0
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,0
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,0


In [3]:
loans_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9578 entries, 0 to 9577
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CreditPolicy    9578 non-null   int64  
 1   Purpose         9578 non-null   object 
 2   IntRate         9578 non-null   float64
 3   Installment     9578 non-null   float64
 4   LogAnnualInc    9578 non-null   float64
 5   Dti             9578 non-null   float64
 6   Fico            9578 non-null   int64  
 7   DaysWithCrLine  9578 non-null   float64
 8   RevolBal        9578 non-null   int64  
 9   RevolUtil       9578 non-null   float64
 10  InqLast6mths    9578 non-null   int64  
 11  Delinq2yrs      9578 non-null   int64  
 12  PubRec          9578 non-null   int64  
 13  NotFullyPaid    9578 non-null   int64  
dtypes: float64(6), int64(7), object(1)
memory usage: 1.0+ MB


In [4]:
# Get all columns of the dataframe
loans_df.columns

Index(['CreditPolicy', 'Purpose', 'IntRate', 'Installment', 'LogAnnualInc',
       'Dti', 'Fico', 'DaysWithCrLine', 'RevolBal', 'RevolUtil',
       'InqLast6mths', 'Delinq2yrs', 'PubRec', 'NotFullyPaid'],
      dtype='object')

## a) Build Logistic Regression Model

### i) Randomly split dataset

In [5]:
# Import sklearn library for splitting dataset and accuracy
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Split the dataset into training and testing datasets
df_train, df_test = train_test_split(loans_df, test_size=0.3, random_state=42)

# Accuracy on test set of a simple baseline model that predicts that all loans will be paid back in full (NotFullyPaid = 0)
y_pred = np.zeros(df_test.shape[0])
y_true = df_test['NotFullyPaid']
accuracy = np.sum(y_pred == y_true) / len(y_true)
print("Accuracy score: ", accuracy)

Accuracy score:  0.83785664578984


After splitting the dataset above into 70% training data and 30% test data, </br>
the accuracy evaluated is: 0.83785664578984

### ii) Logistic Regression Model, Predict NotFullyPaid

In [6]:
# Import logistic regression related libraries
import statsmodels.formula.api as smf

# Create a logistic regression model
logistic_regression_model = smf.logit("NotFullyPaid ~ CreditPolicy + Purpose + IntRate + Installment + LogAnnualInc + Dti \
                                      + Fico + DaysWithCrLine + RevolBal + RevolUtil + InqLast6mths + Delinq2yrs + PubRec" 
                                      , data=df_train)
logistic_regression_model_result = logistic_regression_model.fit()
print(logistic_regression_model_result.summary())

Optimization terminated successfully.
         Current function value: 0.409992
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:           NotFullyPaid   No. Observations:                 6704
Model:                          Logit   Df Residuals:                     6685
Method:                           MLE   Df Model:                           18
Date:                Sun, 27 Oct 2024   Pseudo R-squ.:                 0.06453
Time:                        18:51:58   Log-Likelihood:                -2748.6
converged:                       True   LL-Null:                       -2938.2
Covariance Type:            nonrobust   LLR p-value:                 1.989e-69
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                         8.3393      1.545      5.399      0.

Dependent Variable: NotFullyPaid is the variable the model will be predicting to see whether a loan was not fully paid.
Pseudo R-squared: 0.06453 means that the model explains 6.45% of the variance which is low. A lot of unexplained variance. 
Log-Likelihood: -2748.6 is negative, indicating a better fit. 

Significant Variables (p-values < 0.05):
- Purpose[T.credit_card]
- Purpose[T.debt_consolidation]
- Purpose[T.small_business]
- CreditPolicy
- Installment
- LogAnnualInc
- Fico
- RevolBal
- InqLast6mths
- Delinq2yrs
- PubRec

Signifcant variables contribute meaningfully towards predicting whether or not loan not fully paid. 
The signs of the coefficients will have different meaning. 
- Negative coefficients suggests that the independent variable will have a negative correlation with the dependent variable.
    As the independent variable increases, the less likely the loan is not being fully repaid. 
- Positive coefficients is the other way around. 

Non-Significant Variables (p-values >= 0.05):
- Purpose[T.educational]
- Purpose[T.home_improvement]
- Purpose[T.major_purchase]
- IntRate
- Dti
- DaysWithCrLine
- RevolUtil

Non-Signficant Variables contribute less meaningfully towards predicting the dependedant variable. In the non-significant 
variables above, IntRate is one that stood out. Usually, higher interest rate would mean that a loan is less likely to be 
fully paid but in this statistical findings, IntRate's p-value suggest that it is not significant. 

### iii) Comparing Application A and B

In [7]:
# Logit Formula
# Get Coefficients
coefficients = logistic_regression_model_result.params

# Produce formula
formula = "Logit = " + f"{coefficients['Intercept']:.4f}"  

for var in coefficients.index[1:]: 
    coef_value = coefficients[var]
    formula += f" + ({coef_value:.4f}) * {var}"

print("Logit formula:")
print(formula)

Logit formula:
Logit = 8.3393 + (-0.4272) * Purpose[T.credit_card] + (-0.2896) * Purpose[T.debt_consolidation] + (0.2043) * Purpose[T.educational] + (0.1824) * Purpose[T.home_improvement] + (-0.3555) * Purpose[T.major_purchase] + (0.5067) * Purpose[T.small_business] + (-0.3817) * CreditPolicy + (0.6990) * IntRate + (0.0012) * Installment + (-0.3766) * LogAnnualInc + (0.0026) * Dti + (-0.0090) * Fico + (0.0000) * DaysWithCrLine + (0.0000) * RevolBal + (0.0015) * RevolUtil + (0.0812) * InqLast6mths + (-0.1457) * Delinq2yrs + (0.2757) * PubRec


Since Logit(A) and Logit(B) are identical except for their FICO scores, we can cross out most of the calculations/variables in the equation. 
Application A's FICO: 700
Application B's FICO: 710
Difference in FICO Score = 700 - 710 =

Logit(A) - Logit(B) = (-0.0090) * 700 - ((-0.0090) * 710)
                    = 0.090

### iv) Predict Probability of Test Set Loans

In [8]:
# Add PredictedRisk into test data
df_test['PredictedRisk'] = logistic_regression_model_result.predict(df_test)

# Accuracy using threshold of 0.5
threshold = 0.5
df_test['PredictedNotFullyPaid'] = (df_test['PredictedRisk'] > threshold) * 1

# Calculate accuracy
accuracy = np.sum(df_test['NotFullyPaid'] == df_test['PredictedNotFullyPaid']) / df_test.shape[0]
print("Accuracy score: ", accuracy)

Accuracy score:  0.8382045929018789


Baseline accuracy score:  0.83785664578984

Test set accuracy score: 0.8382045929018789

The accuracy scores of both models differs slightly, to be exact by 0.03% with Logistic Regression model having a higher accuracy. 

## b) Lending Club

### i) Logistic Regression model only with IntRate

In [9]:
# Split the dataset into training and testing datasets
# df_train, df_test = train_test_split(loans_df, test_size=0.3, random_state=42)

# Build the logistic regression model
logistic_regression_model = smf.logit("NotFullyPaid ~ IntRate", data=df_train)
logistic_regression_model_result = logistic_regression_model.fit()

# Show summary of the model
print(logistic_regression_model_result.summary())

Optimization terminated successfully.
         Current function value: 0.426903
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:           NotFullyPaid   No. Observations:                 6704
Model:                          Logit   Df Residuals:                     6702
Method:                           MLE   Df Model:                            1
Date:                Sun, 27 Oct 2024   Pseudo R-squ.:                 0.02594
Time:                        18:51:59   Log-Likelihood:                -2862.0
converged:                       True   LL-Null:                       -2938.2
Covariance Type:            nonrobust   LLR p-value:                 5.098e-35
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -3.5983      0.167    -21.607      0.000      -3.925      -3.272
IntRate       15.3201      1.

IntRate has a p-value of 0 here, which is less than 0.05, means that it is a significant variable. </br>
In the first logistic regression model, the p-value for IntRate is 0.735 and that means tht it is not a significant variable. </br>

Comparing the two logistic regression model, there is a difference in significance of the IntRate variable. </br>
This can be due to how the other variables in the first model interacts with IntRate. Other variables in the </br>
first model could have more effects towards NotFullyPaid compared to IntRate. Furthermore, </br>
in this model, IntRate is the only independent variable and it is signficant on its own. However, this may not be the case</br>
if there are other variables involved, leading to IntRate not having a big effect on predictive power compared to </br>
other variables. 

### ii) Test Set Predictions

In [10]:
# Predict on test set
new_pred = logistic_regression_model_result.predict(df_test['IntRate'])

# Highest predicted probability
highest_prob = new_pred.max()
print("Highest predicted probability: ", highest_prob)

# Threshold of 0.5
threshold = 0.5
new_pred = (new_pred > threshold) * 1
predicted_num_not_paid_back = new_pred.sum()
print("Predicted NotFullyPaid: ", predicted_num_not_paid_back)


Highest predicted probability:  0.4136395278887185
Predicted NotFullyPaid:  0


## c) Identify Profitable Loans

### i) Risk and Reward

In [11]:
def investment_payback(c, r, t):
    return c * np.exp(r * t)

# Vars
c = 10
r = 0.06
t = 3
print(f"Investment payback: ${investment_payback(c, r, t):.2f}")


Investment payback: $11.97


### ii) Profit Calculation

In [12]:
profit_fully_repaid = investment_payback(c, r, t) - c
print(f"Profit if fully repaid: ${profit_fully_repaid:.2f}")

profit_not_fully_repaid = -c
print(f"Profit if not fully repaid: ${profit_not_fully_repaid:.2f}")

Profit if fully repaid: $1.97
Profit if not fully repaid: $-10.00


### iii) $1 Investment

In [13]:
# $1 Investment
c = 1

# Add profit column to test data
df_test['Profit'] = np.where(df_test['NotFullyPaid'] == 0, np.exp(df_test['IntRate'] * 3) - 1, -1)

# Max profit
max_profit = df_test['Profit'].max()
print(f"Max profit: ${max_profit:.2f}")

Max profit: $0.89


### iv) HighInterest

In [14]:
# HighInterest dataset
HighInterest = df_test[df_test['IntRate'] >= 0.15]

# Average profit of $1 investment
average_profit = HighInterest['Profit'].mean()
print(f"Average profit: ${average_profit:.2f}")

# Proportion of high interest loans were not paid back in full
proportion_not_fully_paid = HighInterest['NotFullyPaid'].mean()
print(f"Proportion not fully paid: {proportion_not_fully_paid:.2f}")

Average profit: $0.21
Proportion not fully paid: 0.26


### v) Top 100 loans with smallest values of PredictedRisk

In [15]:
# Sort loans by PredictedRisk
SelectedLoans = HighInterest.sort_values(by='PredictedRisk', ascending=True).head(100)

# Profit of selected loans
profit_selected_loans = SelectedLoans['Profit'].sum()
print(f"Profit of selected loans: ${profit_selected_loans:.2f}")

# Proportion of selected loans that were not paid back in full
proportion_not_fully_paid_selected = SelectedLoans['NotFullyPaid'].sum()
print(f"Proportion not fully paid of selected loans: {proportion_not_fully_paid_selected}")

# Comparison with simple strategy
profit_simple_strategy = 20.94

Profit of selected loans: $36.38
Proportion not fully paid of selected loans: 16


Looking at the top 100 loans with high interest and smallest predicted risk, the profit calculated is: $36.38
Proportion of the loans that was not fully paid is 16 out of 100. 

Comparing to the simple strategy of investing in all loans and yield a profit of $20.94 for $100 investment, this yields a </br>
20.94% of return. The selected loans yield a return of $36.38, which is more profitable, higher yield of profit return. 

## d) Improvements as an Analyst

Predictive modelling takes in a set of data that was recorded in the past and use it to predict the future. This brings in the assumption that the relationship between variables remain stable. However, this is not the case, especially with financial situations. There will be other factors involved that will disrupt this assumption of a stable relationship. Some of the external factors to consider could be:
- Recession
- Change in behavior
- Political effects

Analyst Improvements:
- In financial situations, there are a lot of cycles involved, and we can continue to use predictive modelling but make sure to retrain the model periodically. 
- In the data set we can introduce more macroeconomical variables like unemployment and inflation rate. Model will then have better adaptations towards these changes. 
- Maintain consistent monitoring on model's performance and notify whenever a performance is below a certain threshold for further investigation such as retraining or updating data. 